# Домашнее задание 5: QLoRA-дообучение causal LM

In [1]:
%pip install -q transformers datasets accelerate peft bitsandbytes trl
%pip install -q lm-eval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 8.3 MB/s eta 0:00:00


In [2]:
import gc
import os
import random
import subprocess

import numpy as np
import torch
from datasets import Dataset, load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from torch.profiler import ProfilerActivity, profile, record_function
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

# Фиксация окружения

In [3]:
import platform
from importlib.metadata import version

print("python:", platform.python_version())
print("platform:", platform.platform())
print("torch:", torch.__version__)
print("transformers:", version("transformers"))
print("datasets:", version("datasets"))
print("peft:", version("peft"))
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda_version:", torch.version.cuda)
    print("gpu:", torch.cuda.get_device_name(0))


python: 3.12.13
platform: Linux-6.6.113+-x86_64-with-glibc2.35
torch: 2.10.0+cu128
transformers: 5.0.0
datasets: 4.8.4
peft: 0.18.1
cuda_available: True
cuda_version: 12.8
gpu: Tesla T4


# 1. Выбор датасета

## Воспроизводимость и данные

Взял `allenai/c4`, потому что это большой и разнообразный корпус обычного веб-текста: на нем удобно учить базовую языковую модель без узкой доменной привязки

Плюс `streaming`-режим практичный: можно брать контролируемый срез и не упираться в память

Фиксируем сиды и загружаем срез `allenai/c4`

In [4]:
RANDOM_STATE = 42


def set_seed(seed: int = RANDOM_STATE) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(RANDOM_STATE)

In [5]:
from itertools import islice
from google.colab import userdata
from huggingface_hub import login

C4_LANG = "en"
n_rows = 20_000

take = n_rows if n_rows is not None else 20_000
stream = load_dataset(
    "allenai/c4",
    C4_LANG,
    split="train",
    streaming=True,
)
raw_ds = Dataset.from_list(list(islice(stream, take)))
raw_ds = raw_ds.shuffle(seed=RANDOM_STATE)
print(len(raw_ds), raw_ds[0]["text"][:500])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1024 [00:00<?, ?it/s]

20000 We've invested time, resources, and talent into the technology, partnerships, and processes we developed. We are proud of the what we've built and we want our clients to be confident working with First Look which is why every Appraisal we deliver is backed by our Quality and Performance Guarantee for qualified clients.
We've invested time, resources, and talent into the technology, partnerships, and processes we developed.


Датасет загружен в streaming-режиме, срез зафиксирован и готов к токенизации и обучению

# 2. Выбор предобученной модели

## Модель и токенизация

Выбрал Qwen2.5-1.5B: модель попадает в требуемый размер, хорошо работает как базовая для дообучения и дает адекватное качество на старте. Для экономии памяти загружаем ее в 4-bit режиме и дальше обучаем через QLoRA.

In [6]:
MODEL_ID = "Qwen/Qwen2.5-1.5B"
max_len = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tok(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_len,
        padding=False,
    )


tokenized = raw_ds.map(
    tok,
    batched=True,
    remove_columns=raw_ds.column_names,
)
print(tokenized)
print(tokenizer.decode(tokenized[0]["input_ids"][:120]))

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 20000
})
We've invested time, resources, and talent into the technology, partnerships, and processes we developed. We are proud of the what we've built and we want our clients to be confident working with First Look which is why every Appraisal we deliver is backed by our Quality and Performance Guarantee for qualified clients.
We've invested time, resources, and talent into the technology, partnerships, and processes we developed.


## 4-bit загрузка модели

Грузим базу в QLoRA-режиме (NF4) и готовим её к k-bit training

In [7]:
assert torch.cuda.is_available()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

# 3. Предварительная оценка качества

## Генерация до обучения

Собираем baseline на фиксированной предметной корзинке из того же распределения (`c4`) и на одинаковых параметрах генерации

In [ ]:
basket_prompts = [
    "The main idea of renewable energy is",
    "In machine learning, overfitting means",
    "Software engineering best practices include",
]

gen_before = {}
dev = next(model.parameters()).device


def sample_generate(prompt: str, max_new: int = 64) -> str:
    model.eval()
    batch = tokenizer(prompt, return_tensors="pt").to(dev)
    with torch.no_grad():
        out = model.generate(
            **batch,
            max_new_tokens=max_new,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)


for p in basket_prompts:
    gen_before[p] = sample_generate(p)
    print("---", p, "\n", gen_before[p][:400], "\n")

--- The main idea of renewable energy is 
 The main idea of renewable energy is to use energy that is available in nature and will never run out. The main sources of renewable energy are solar, wind, hydro, geothermal, and biomass. The main sources of non-renewable energy are coal, oil, and natural gas. The main sources of nuclear energy are uranium and plutonium. The 

--- In machine learning, overfitting means 
 In machine learning, overfitting means that a model learns the training data too well, to the point where it performs poorly on new, unseen data. This is a problem because it means that the model is too complex and has learned the noise in the training data instead of the underlying pattern.
There are several ways to prevent overfitting, including regularization, 

--- Software engineering best practices include 
 Software engineering best practices include the following: ____.
A. Modular design
B. Object-oriented design
C. Object-oriented programming
D. Object-oriented testi

# 4. QLoRA-дообучение

## LoRA + обучение

##### Запускаем обучение через Trainer

##### Параметры выбраны простые и безопасные для T4: r=16 и alpha=32 как баланс качества и памяти, learning rate 1e-4 как умеренный шаг, длина 512 чтобы держать контекст и не упереться в VRAM. Обучаем 250 шагов с проверкой каждые 50 шагов и берем лучший чекпоинт по eval loss

##### Ожидание: небольшое улучшение связности без заметной деградации

In [9]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [10]:
splits = tokenized.train_test_split(test_size=0.05, seed=RANDOM_STATE)
train_ds = splits["train"]
eval_ds = splits["test"]

collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

args = TrainingArguments(
    output_dir="./c4_qlora_out",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_steps=250,
    warmup_steps=25,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    gradient_checkpointing=True,
    report_to="none",
    seed=RANDOM_STATE,
)

model.enable_input_require_grads()
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=collator,
)
trainer.train()

history_rows = []
for row in trainer.state.log_history:
    if "loss" in row or "eval_loss" in row:
        history_rows.append(
            {
                "step": row.get("step"),
                "loss": row.get("loss"),
                "eval_loss": row.get("eval_loss"),
            }
        )

print("step | train_loss | eval_loss")
for row in history_rows:
    step = row["step"]
    train_loss = "-" if row["loss"] is None else f"{row['loss']:.4f}"
    eval_loss = "-" if row["eval_loss"] is None else f"{row['eval_loss']:.4f}"
    print(f"{step:>4} | {train_loss:>10} | {eval_loss:>9}")

trainer.model.save_pretrained("./c4_qlora_adapter")
tokenizer.save_pretrained("./c4_qlora_adapter")

Step,Training Loss,Validation Loss
50,2.644591,2.806428
100,2.764737,2.790070
150,2.791174,2.785317
200,2.732645,2.783340
250,2.777941,2.782963


step | train_loss | eval_loss
  10 |     2.7879 |         -
  20 |     2.7751 |         -
  30 |     2.8826 |         -
  40 |     2.7817 |         -
  50 |     2.6446 |         -
  50 |          - |    2.8064
  60 |     2.8031 |         -
  70 |     2.7680 |         -
  80 |     2.7169 |         -
  90 |     2.7010 |         -
 100 |     2.7647 |         -
 100 |          - |    2.7901
 110 |     2.6715 |         -
 120 |     2.7929 |         -
 130 |     2.8360 |         -
 140 |     2.6894 |         -
 150 |     2.7912 |         -
 150 |          - |    2.7853
 160 |     2.6974 |         -
 170 |     2.7551 |         -
 180 |     2.8830 |         -
 190 |     2.6866 |         -
 200 |     2.7326 |         -
 200 |          - |    2.7833
 210 |     2.7346 |         -
 220 |     2.7614 |         -
 230 |     2.8057 |         -
 240 |     2.7704 |         -
 250 |     2.7779 |         -
 250 |          - |    2.7830


('./c4_qlora_adapter/tokenizer_config.json',
 './c4_qlora_adapter/chat_template.jinja',
 './c4_qlora_adapter/tokenizer.json')

Собрали динамику train loss и eval loss по шагам. По этим значениям можно отследить переобучение, если eval loss начинает расти при падении train loss

# 5. Оценка качества после обучения

## Сравнение генерации до/после

 Берем ту же предметную корзинку и смотрим, что изменилось после обучения

In [ ]:
gen_after = {}
for p in basket_prompts:
    gen_after[p] = sample_generate(p)

for p in basket_prompts:
    print(p)
    print("До:   ", gen_before[p][:350].replace("\n", " "))
    print("После:", gen_after[p][:350].replace("\n", " "))
    print()

The main idea of renewable energy is
До:    The main idea of renewable energy is to use energy that is available in nature and will never run out. The main sources of renewable energy are solar, wind, hydro, geothermal, and biomass. The main sources of non-renewable energy are coal, oil, and natural gas. The main sources of nuclear energy are uranium and plutonium. The
После: The main idea of renewable energy is to use energy that is available in nature and is not limited. The main advantage of renewable energy is that it is not limited and can be used for a long time. The main disadvantage of renewable energy is that it is not as efficient as non-renewable energy. The main advantage of renewable energy is that it is

In machine learning, overfitting means
До:    In machine learning, overfitting means that a model learns the training data too well, to the point where it performs poorly on new, unseen data. This is a problem because it means that the model is too complex and has learned

Сравнили генерацию до и после обучения на одинаковых примерах и в одинаковых настройках

## lm-eval после дообучения

Короткий бенчмарк для модели с подключенным адаптером `peft`

In [12]:
%pip install -q lm-eval

import subprocess

model_args = f"pretrained={MODEL_ID},peft=./c4_qlora_adapter"
after_path = "./lm_eval_after.json"
EVAL_LIMIT = 200

subprocess.run(
    [
        "lm_eval",
        "--model",
        "hf",
        "--model_args",
        model_args,
        "--tasks",
        "lambada_openai",
        "--device",
        "cuda",
        "--batch_size",
        "auto",
        "--limit",
        str(EVAL_LIMIT),
        "--output_path",
        after_path,
    ],
    check=False,
)

print(f"Saved post-train lm-eval to: {after_path} (limit={EVAL_LIMIT})")

Saved post-train lm-eval to: ./lm_eval_after.json (limit=200)


# 6. Профайлинг обучения

## Профайлинг шага обучения

Смотрим, где модель реально тратит время на forward/backward/optimizer

In [13]:
model.train()
trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=2e-4)
dev = next(model.parameters()).device

rows = [tokenized[i] for i in range(0, min(4, len(tokenized)))]
batch = collator(rows)
batch = {k: v.to(dev) for k, v in batch.items()}

with profile(
    activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA],
    record_shapes=True,
    profile_memory=True,
) as prof:
    for step in range(3):
        with record_function("forward"):
            out = model(**batch)
        loss = out.loss
        with record_function("backward"):
            loss.backward()
        with record_function("optimizer"):
            opt.step()
            opt.zero_grad(set_to_none=True)
        prof.step()

print(
    prof.key_averages().table(
        sort_by="cuda_time_total",
        row_limit=15,
    )
)
model.eval()

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:217: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg       CPU Mem  Self CPU Mem      CUDA Mem  Self CUDA Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                forward         0.00%       0.000us         0.00%       0.000us       0.000us        2.551s        74.21%        2.551s     850.491ms           0 B           0 B           0 B           0 

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

### Вывод по профайлеру

Основное время уходит в forward/backward и матричные операции на GPU. Также есть заметные накладные расходы на копирования и на 4-bit quant/dequant. Ускорять в первую очередь стоит через batch, длину последовательности, checkpointing и более удачный kernel-стек.

### Бейзлайн на lm-eval

Это прогон той же задачи для базовой модели без LoRA. Результат тоже сохраняется в JSON, чтобы потом сравнить числа напрямую.

In [14]:
model_args_base = f"pretrained={MODEL_ID}"
base_path = "./lm_eval_base.json"
EVAL_LIMIT = 200

subprocess.run(
    [
        "lm_eval",
        "--model",
        "hf",
        "--model_args",
        model_args_base,
        "--tasks",
        "lambada_openai",
        "--device",
        "cuda",
        "--batch_size",
        "auto",
        "--limit",
        str(EVAL_LIMIT),
        "--output_path",
        base_path,
    ],
    check=False,
)

print(f"Saved baseline lm-eval to: {base_path} (limit={EVAL_LIMIT})")

Saved baseline lm-eval to: ./lm_eval_base.json (limit=200)


# Сравнение метрик lm-eval

Фактический прогон на lambada_openai, limit=200.

Baseline: perplexity 5.8798, acc 0.61

После LoRA: perplexity 6.1754, acc 0.60

## Разница после дообучения:

perplexity +0.2956 (хуже, так как меньше лучше), acc -0.01

## Вывод: явного улучшения не видно, качество немного просело

# Итоговые выводы

После LoRA метрики слегка просели. Профайлер показывает, что узкое место в compute на GPU

## Почему могло стать хуже

Возможны четыре причины: мало шагов, различие между обучающей и тестовой задачей, неидеальные lr и LoRA-параметры, и шум из-за малого eval limit

## Что проверить дальше

Увеличить шаги и объем данных, прогнать небольшую сетку lr и r/alpha, добавить еще 1-2 задачи в lm-eval и повторить прогон на нескольких сидах